# 3 Train, evaluate and deploy (run in SageMaker Studio)

Consolidates the former sections 4.0 benchmark, 4.1 XGBoost training, 4.2 evaluation (with the manual approval gate) and 5.0 batch deployment. Run `2_data_pipeline.ipynb` first. Billable steps are marked in the section headings.

In [ ]:
# SageMaker Python SDK v3 (default on current Studio images) removed the v2 modules this notebook uses.
# Pin v2; on first run this installs it and restarts the kernel - then run the notebook again from the top.
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

## Bootstrap

Path, config, boto3 session/clients and shared S3 / batch-transform helpers.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json
import boto3, numpy as np, pandas as pd
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_parquet(name):
    b, k = split_key(SPLITS + name)
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
def read_json(uri):
    b, k = split_key(uri)
    return json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
CLASSES = list(range(K))
def class_name(i):
    return ACTIVITY_NAMES[LABEL_MAP[int(i)]]
print(K, "classes;", len(FEATURES), "features")

In [ ]:
import sagemaker
from src.pamap2.features import intensity_of
sm = session.client("sagemaker")
sm_session = sagemaker.Session(boto_session=session)
import time
from sagemaker.transformer import Transformer

def upload_features_csv(df, key):
    """Header-less, feature-only CSV in feature_columns.json order."""
    E.check_feature_columns(df, FEATURES)
    body = df[FEATURES].to_csv(header=False, index=False).encode()
    uri = s3_uri(cfg, "models", key)
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=body)
    return uri

def run_transform(model_name, input_uri, output_prefix, join=False):
    """Batch transform on ml.m5.large x1; waits for completion. Nothing stays running afterwards."""
    t = Transformer(model_name=model_name, instance_count=1, instance_type="ml.m5.large",
                    strategy="MultiRecord", assemble_with="Line", accept="text/csv",
                    output_path=output_prefix, sagemaker_session=sm_session)
    t.transform(data=input_uri, content_type="text/csv", split_type="Line",
                join_source="Input" if join else "None", wait=True, logs=False)
    return output_prefix + input_uri.rsplit("/", 1)[1] + ".out"

def read_probs(out_uri, k):
    b, kk = split_key(out_uri)
    return E.parse_probs(s3.get_object(Bucket=b, Key=kk)["Body"].read(), k)

## 4.0 Benchmark (run in SageMaker Studio)

Simple baseline for Part C: a depth-3 decision tree (scikit-learn, trained locally) on two features - chest acc16 magnitude std and HR mean (NaN filled with train median). A majority-class row is reported as the floor. Primary metric: macro-F1 on test. Test scores are optimistic (seen subjects, per spec B).

### Load splits and fit

In [ ]:
from src.pamap2.benchmark import BENCHMARK_FEATURES, fit_benchmark, predict_benchmark, predict_majority
train, val, test = (read_parquet(f"{n}.parquet") for n in ("train", "validation", "test"))
for df in (train, val, test):
    E.check_feature_columns(df, FEATURES)
model = fit_benchmark(train)
print("Features:", BENCHMARK_FEATURES, "| train medians:", model["medians"], "| majority class:", class_name(model["majority"]))

### Metrics on validation and test

In [ ]:
report = {"features": BENCHMARK_FEATURES, "model": "DecisionTree(max_depth=3)",
          "label_map": {str(k): v for k, v in LABEL_MAP.items()}}
for name, df in (("validation", val), ("test", test)):
    for label, pred in (("tree", predict_benchmark(model, df)), ("majority", predict_majority(model, df))):
        report[f"{name}_{label}"] = E.classification_metrics(df["label"], pred, CLASSES)
summary = pd.DataFrame([{"split": s, "model": m, "macro_f1": report[f"{s}_{m}"]["macro_f1"],
                         "accuracy": report[f"{s}_{m}"]["accuracy"]}
                        for s in ("validation", "test") for m in ("tree", "majority")])
display(summary)

In [ ]:
pc = report["test_tree"]["per_class"]
display(pd.DataFrame({class_name(k): v for k, v in pc.items()}).T)

### Save report

In [ ]:
write_json(REPORTS + "benchmark.json", report)

## 4.1 Train XGBoost with SageMaker Debugger (run in SageMaker Studio)

Built-in SageMaker XGBoost 1.7-1, `multi:softprob`, fixed hyperparameters (no HPO), unweighted, early stopping on validation. One `ml.m5.xlarge`. Debugger rules: LossNotDecreasing, Overfit, ClassImbalance + the XGBoost report. The model is registered in the Model Registry as `PendingManualApproval`.

In [ ]:
from sagemaker.inputs import TrainingInput
from sagemaker.debugger import Rule, rule_configs, DebuggerHookConfig, CollectionConfig
ROLE = cfg["role_arn"]
image = sagemaker.image_uris.retrieve("xgboost", cfg["region"], cfg["xgboost_version"])
print(image)

### Estimator, hyperparameters, Debugger

In [ ]:
output_path = s3_uri(cfg, "models", "xgboost/")
estimator = sagemaker.estimator.Estimator(
    image_uri=image, role=ROLE, instance_count=1, instance_type="ml.m5.xlarge",
    output_path=output_path, sagemaker_session=sm_session,
    base_job_name=f"{cfg['prefix']}-xgboost",
    max_run=3600,
    debugger_hook_config=DebuggerHookConfig(
        s3_output_path=s3_uri(cfg, "models", "xgboost/debugger/"),
        collection_configs=[CollectionConfig(name=n, parameters={"save_interval": "5"})
                            for n in ("metrics", "feature_importance", "average_shap",
                                      "labels", "predictions")]),
    rules=[Rule.sagemaker(rule_configs.loss_not_decreasing()),
           Rule.sagemaker(rule_configs.overfit()),
           Rule.sagemaker(rule_configs.class_imbalance()),
           Rule.sagemaker(rule_configs.create_xgboost_report())],
)
estimator.set_hyperparameters(
    objective="multi:softprob", num_class=K, num_round=500, early_stopping_rounds=20,
    eval_metric="mlogloss,merror", max_depth=6, eta=0.1, subsample=0.8, colsample_bytree=0.8)

### Train (billable: ~one ml.m5.xlarge job)

In [ ]:
estimator.fit({"train": TrainingInput(SPLITS + "train.csv", content_type="text/csv"),
               "validation": TrainingInput(SPLITS + "validation.csv", content_type="text/csv")}, wait=True)
job = estimator.latest_training_job.name
print("Training job:", job, "| model:", estimator.model_data)

### Debugger rule statuses and report

In [ ]:
summary = estimator.latest_training_job.rule_job_summary()
display(pd.DataFrame([{k: r.get(k) for k in ("RuleConfigurationName", "RuleEvaluationStatus", "StatusDetails")}
                      for r in summary]))
print("Debugger artifacts:", estimator.latest_job_debugger_artifacts_path())
print("XGBoost report:", estimator.latest_job_debugger_artifacts_path() + "rule-output/CreateXgboostReport/")

### Register in Model Registry (PendingManualApproval)

In [ ]:
group = cfg["model_package_group"]
try:
    sm.describe_model_package_group(ModelPackageGroupName=group)
except sm.exceptions.ClientError as e:
    if "does not exist" not in str(e) and "ValidationException" not in str(e):
        raise
    sm.create_model_package_group(ModelPackageGroupName=group,
                                  ModelPackageGroupDescription="PAMAP2 HAR XGBoost models")
pkg = estimator.register(content_types=["text/csv"], response_types=["text/csv"],
                         inference_instances=["ml.m5.large"], transform_instances=["ml.m5.large"],
                         model_package_group_name=group, approval_status="PendingManualApproval",
                         description="PAMAP2 activity recognition, multi:softprob")
pkg_arn = pkg.model_package_arn
final = {m["MetricName"]: m["Value"] for m in sm.describe_training_job(TrainingJobName=job).get("FinalMetricDataList", [])}
print(final)
sm.update_model_package(ModelPackageArn=pkg_arn, CustomerMetadataProperties={
    **{k.replace(":", "_"): f"{v:.6f}" for k, v in final.items()}, "training_job": job})
print("Registered:", pkg_arn)

In [ ]:
write_json(REPORTS + "xgboost_training.json", {
    "training_job": job, "model_data": estimator.model_data, "image_uri": image,
    "model_package_arn": pkg_arn, "final_metrics": final,
    "debugger_artifacts": estimator.latest_job_debugger_artifacts_path()})

## 4.2 Evaluate XGBoost vs benchmark (run in SageMaker Studio)

Batch transform on the test split, metrics (macro-F1 headline), comparison with the benchmark and a confusion matrix. **Test scores are optimistic**: test windows come from subjects seen in training (chronological split, spec B). The last cell is a manual approval step - run it only if you are satisfied.

### Create a temporary SageMaker model from the trained artifact

In [ ]:
train_info = read_json(REPORTS + "xgboost_training.json")
eval_model_name = f"{cfg['prefix']}-eval-{int(time.time())}"
sm.create_model(ModelName=eval_model_name, ExecutionRoleArn=cfg["role_arn"],
                PrimaryContainer={"Image": train_info["image_uri"], "ModelDataUrl": train_info["model_data"]})
test = read_parquet("test.parquet")
test_uri = upload_features_csv(test, "transform-input/test/features.csv")
print(len(test), "test rows ->", test_uri)

### Batch transform (billable: one ml.m5.large job, shuts down on completion)

In [ ]:
try:
    out_uri = run_transform(eval_model_name, test_uri, s3_uri(cfg, "models", "transform-output/test/"))
    probs = read_probs(out_uri, K)
    assert len(probs) == len(test), (len(probs), len(test))
    pred = E.probs_to_labels(probs, K)
    metrics = E.classification_metrics(test["label"], pred, CLASSES)
finally:
    sm.delete_model(ModelName=eval_model_name)   # only the temporary model created above

### Benchmark vs XGBoost

In [ ]:
bench = read_json(REPORTS + "benchmark.json")["test_tree"]
cmp = E.compare(bench, metrics)
print(f"HEADLINE macro-F1 (test): benchmark={bench['macro_f1']:.4f}  xgboost={metrics['macro_f1']:.4f}  better={cmp['better']}")
display(cmp["table"])
display(pd.DataFrame({class_name(k): v for k, v in metrics["per_class"].items()}).T)

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(metrics["confusion_matrix"])
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
names = [class_name(i) for i in CLASSES]
ax.set_xticks(range(K)); ax.set_xticklabels(names, rotation=90)
ax.set_yticks(range(K)); ax.set_yticklabels(names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("XGBoost test confusion matrix (row-normalised)")
fig.colorbar(im); plt.tight_layout(); plt.show()

### Save report

In [ ]:
write_json(REPORTS + "xgboost_eval.json", {**metrics, "label_map": {str(k): v for k, v in LABEL_MAP.items()},
    "benchmark_macro_f1": bench["macro_f1"], "better_than_benchmark": cmp["better"],
    "model_package_arn": train_info["model_package_arn"]})
sm.update_model_package(ModelPackageArn=train_info["model_package_arn"], CustomerMetadataProperties={
    "test_macro_f1": f"{metrics['macro_f1']:.6f}", "test_accuracy": f"{metrics['accuracy']:.6f}",
    "benchmark_macro_f1": f"{bench['macro_f1']:.6f}", "eval_report": REPORTS + "xgboost_eval.json",
    **{k.replace(":", "_"): f"{v:.6f}" for k, v in train_info["final_metrics"].items()}})

### MANUAL APPROVAL - human gate: set `APPROVE = True` only if you are satisfied with the comparison above

There is no scripted gate on the metrics. With `APPROVE = False` (the default) the approval cell only prints a message, nothing is approved, and section 5.0 stops because the package is not `Approved`. Setting `APPROVE = True` and running the next cells marks the model package `Approved`, which lets section 5.0 deploy it.

In [ ]:
APPROVE = False   # human decision: set to True only after reviewing the comparison above

In [ ]:
# MANUAL STEP: run only after reviewing the benchmark comparison (approves only when APPROVE is True).
if not APPROVE:
    print("APPROVE is False - model package NOT approved: " + train_info["model_package_arn"] + chr(10) +
          "Review the comparison above, set APPROVE = True and re-run this cell to approve.")
else:
    if not cmp["better"]:
        print("!" * 70 + chr(10) + "WARNING: XGBoost did NOT beat the benchmark on test macro-F1. Approving anyway (no auto-block)." + chr(10) + "!" * 70)
    sm.update_model_package(ModelPackageArn=train_info["model_package_arn"], ModelApprovalStatus="Approved",
                            ApprovalDescription="Approved manually after benchmark comparison in 4.2")
    print("Approved:", train_info["model_package_arn"])

## 5.0 Deploy: batch transform on the production split (run in SageMaker Studio)

Deployment is batch transform only (no real-time endpoint, nothing left running). Uses the exact package ARN recorded by 4.2 in `xgboost_eval.json`, which must be Approved. Input/output join is enabled (output rows = input features + class probabilities) so Part D can run batch-mode monitoring on production outputs. Production labels are not used for training; `true_activity_id` is carried through only for later quality monitoring.

### Create model from the latest approved package

In [ ]:
pkg_arn = read_json(REPORTS + "xgboost_eval.json")["model_package_arn"]   # exactly the package 4.2 evaluated
status = sm.describe_model_package(ModelPackageName=pkg_arn)["ModelApprovalStatus"]
if status != "Approved":
    raise RuntimeError(f"{pkg_arn} is {status}, not Approved - deploy stopped. Set APPROVE = True in the "
                       "manual approval section (end of 4.2) and run it first.")
prod_model_name = f"{cfg['prefix']}-prod-{int(time.time())}"
sm.create_model(ModelName=prod_model_name, ExecutionRoleArn=cfg["role_arn"],
                Containers=[{"ModelPackageName": pkg_arn}])
prod = read_parquet("production.parquet").reset_index(drop=True)
prod_uri = upload_features_csv(prod, "transform-input/production/features.csv")
print("Package:", pkg_arn, "|", len(prod), "production rows")

### Batch transform with input/output join (billable: one ml.m5.large job)

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
try:
    out_uri = run_transform(prod_model_name, prod_uri, s3_uri(cfg, "models", f"transform-output/production/{stamp}/"), join=True)
    probs = read_probs(out_uri, K)
    assert len(probs) == len(prod), (len(probs), len(prod))
finally:
    sm.delete_model(ModelName=prod_model_name)   # only the temporary model created above
print("Joined output:", out_uri)

### Write predictions

In [ ]:
cls = E.probs_to_labels(probs, K)
act = [LABEL_MAP[int(c)] for c in cls]
preds = pd.DataFrame({
    "window_id": prod["window_id"].astype(str), "subject_id": prod["subject_id"],
    "predicted_class": cls, "predicted_activity_id": act,
    "activity_name": [ACTIVITY_NAMES.get(a) for a in act],
    "intensity": [intensity_of(a) for a in act],
    "confidence": probs.max(axis=1),
    "true_activity_id": prod["activity_id"],
    "model_package_arn": pkg_arn,
})
assert len(preds) == len(prod) and preds["activity_name"].notna().all()
buf = io.BytesIO(); preds.to_parquet(buf, index=False)
b, k = split_key(s3_uri(cfg, "models", f"predictions/production/predictions-{stamp}.parquet"))
s3.put_object(Bucket=b, Key=k, Body=buf.getvalue())
print(f"Wrote s3://{b}/{k}")
print(preds["activity_name"].value_counts().to_string())
print(preds["intensity"].value_counts().to_string())

### Confirm nothing is left running

In [ ]:
print("Endpoints:", [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]])
print("Transform jobs in progress:", len(sm.list_transform_jobs(StatusEquals="InProgress")["TransformJobSummaries"]))